# Análisis exploratorio de datos

## Carga de datos

In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
import math
import datetime
from scipy.stats import boxcox
from statsmodels.tsa.stattools import adfuller
from pathlib import Path


sns.set_style("darkgrid")
sns.set_context("poster")

In [5]:
pd.set_option('display.max_columns', None) # Por si se quiere ver todas las columnas del DataFrame
pd.set_option('display.max_rows', None) # Por si se quiere ver todas las filas del DataFrame
pd.set_option('display.max_colwidth', None) # Por si se quiere ver todo el contenido de las columnas del DataFrame

In [7]:
DATA_PATH_GEN_DIARIA = Path("../data/raw/ree_generacion_diaria.csv")
DATA_PATH_HORARIO = Path("../data/raw/ree_horario.csv")

if not DATA_PATH_GEN_DIARIA.exists():
    # Alternativa si se ejecuta desde la raíz del proyecto
    DATA_PATH_GEN_DIARIA = Path("data/raw/ree_generacion_diaria.csv")

if not DATA_PATH_HORARIO.exists():
    # Alternativa si se ejecuta desde la raíz del proyecto
    DATA_PATH_HORARIO = Path("data/raw/ree_horario.csv")

df_horario = pd.read_csv(DATA_PATH_HORARIO)
df_generacion = pd.read_csv(DATA_PATH_GEN_DIARIA)

print(
    f"✓ Dataset cargado correctamente con {df_horario.shape[0]} filas y {df_horario.shape[1]} columnas a partir de ree_horario.csv."
)
print(
    f"✓ Dataset cargado correctamente con {df_generacion.shape[0]} filas y {df_generacion.shape[1]} columnas a partir de ree_generacion_diaria.csv."
)
assert df_horario.shape == (
    26280,
    6,
), f"Dimensiones inesperadas: {df_horario.shape} (se esperaban 26280 filas y 6 columnas)"

assert df_generacion.shape == (
    1095,
    18,
), f"Dimensiones inesperadas: {df_generacion.shape} (se esperaban 1095 filas y 18 columnas)"

df_horario.head()

✓ Dataset cargado correctamente con 26280 filas y 6 columnas a partir de ree_horario.csv.
✓ Dataset cargado correctamente con 1095 filas y 18 columnas a partir de ree_generacion_diaria.csv.


,fecha_hora,demanda_prevista,demanda_programada,demanda_real,precio_spot,pvpc
0,2021-01-01 00:00:00+01:00,26104.667,26013.0,26151.667,50.87,NaN
1,2021-01-01 01:00:00+01:00,24822.583,24630.0,24886.250,48.19,NaN
2,2021-01-01 02:00:00+01:00,22652.667,22696.0,22685.500,44.68,NaN
3,2021-01-01 03:00:00+01:00,20786.833,20756.0,20828.250,38.50,NaN
4,2021-01-01 04:00:00+01:00,19877.917,19800.0,19877.583,36.80,NaN


In [8]:
df_generacion.head()

,fecha,gen_hidráulica,gen_nuclear,gen_carbón,gen_motores_diésel,gen_turbina_de_gas,gen_turbina_de_vapor,gen_ciclo_combinado,gen_hidroeólica,gen_eólica,gen_solar_fotovoltaica,gen_solar_térmica,gen_otras_renovables,gen_cogeneración,gen_residuos_no_renovables,gen_residuos_renovables,gen_generación_total,gen_fuel_gas
0,2021-01-01,111077.579,170966.633,5663.327,6180.245,840.614,3681.076,61416.329,15.538,240290.824,29876.523,3126.095,13234.386,54581.986,4477.651,1232.274,706661.080,NaN
1,2021-01-02,112211.332,170974.655,5611.373,7074.243,722.388,4587.264,80555.288,1.957,270599.158,34247.163,6340.881,12636.823,67513.021,4389.159,1262.089,778726.794,NaN
2,2021-01-03,116969.983,171000.851,5597.198,7036.465,993.803,4502.216,83628.611,15.593,221167.693,37134.485,7441.714,12215.001,73185.396,4896.856,1250.171,747036.037,NaN
3,2021-01-04,165965.559,170871.508,8431.686,7371.259,1328.929,3783.796,144897.990,34.367,130542.499,26307.981,2783.566,12124.513,77455.663,5155.834,1215.017,758270.167,NaN
4,2021-01-05,176184.581,170882.098,16295.678,7285.896,1006.802,4453.566,138876.901,-3.059,70360.924,29746.352,2099.695,12386.598,80838.906,5499.507,1230.110,717144.555,NaN


Se comienza analizando `df_horario`, ya que contiene la variable `demanda_real`, que es la que se busca predecir. Sin embargo, se puede observar que viene en horas, mientras que el segundo fichero viene en días, por lo que será necesario más adelante una conversión a días.

In [9]:
df_horario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26280 entries, 0 to 26279
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fecha_hora          26280 non-null  object 
 1   demanda_prevista    26172 non-null  float64
 2   demanda_programada  26172 non-null  float64
 3   demanda_real        26178 non-null  float64
 4   precio_spot         26280 non-null  float64
 5   pvpc                22657 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.2+ MB


In [10]:
nulos = df_horario.isnull().sum()
print(nulos)

fecha_hora               0
demanda_prevista       108
demanda_programada     108
demanda_real           102
precio_spot              0
pvpc                  3623
dtype: int64


Al encontrarse valores nulos en el dataset de horario, no se debería hacer una conversión a días directamente sin haber tratado estos nulos previamente, ya que al realizar la suma se perdería información en las horas faltantes, lo que resulta crítico en `demanda_real`, que es la variable objetivo.